# EXP027: MFI + MLE on real responses (Python)

`data/LNIRT_CredentialForm1/` の実回答に MFI と最尤推定（MLE）を適用します。
各 step で、現在の能力推定値における Fisher 情報量が最大の未出題項目を
受検者ごとに選択し、選択項目への応答を実回答行列から取得します。

初期能力値、MLE、全問正解・全問不正解時の更新、seed、出力形式は、
EXP027 の DQN / MFI notebook と揃えています。これにより、同じ実データ上で
DQN と MFI の step 別 Bias、RMSE、MAE を比較できます。


In [7]:
# -*- coding: utf-8 -*-
from dataclasses import dataclass
from pathlib import Path
from typing import Any, cast

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT = find_project_root()
RESULTS_DIR = ROOT / "EXP027" / "results"

print(f"Project root: {ROOT}")
print(f"Results dir : {RESULTS_DIR}")

Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results


In [8]:
@dataclass
class Config:
    test_length: int = 40

    # Real-response dataset / evaluation size
    dataset: str = "LNIRT_CredentialForm1"
    testing_size: int = 0  # 0: use every row in the testing files

    # Same initial-theta seed as the EXP027 DQN test
    seed: int = 20260430

In [9]:
# Aligned with the MLE and Fisher-information definitions in EXP027.
def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(
            Any,
            minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"),
        )
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)

In [ ]:
def choose_mfi(item_bank, theta_current, item_ids):
    # Choose each examinee's unadministered item with maximum FI.
    information = np.vstack([FI(item_bank, theta) for theta in theta_current])

    if item_ids.shape[0] > 0:
        subject_indices = np.arange(len(theta_current))[:, None]
        information[subject_indices, item_ids.T] = -np.inf

    return information.argmax(axis=1).astype(np.int64)


# Clip ability estimates to the MLE optimisation bounds so the all-correct /
# all-incorrect fallback cannot drift past the estimation range toward the
# extreme (winsorized) item difficulties (e.g. b_min ~ -6). Matches EXP027 DQN.
THETA_MIN, THETA_MAX = -4.0, 4.0


def estimate_theta_mle(item_bank, item_ids, responses, current_theta):
    testing_size = responses.shape[1]
    theta_hat = np.zeros(testing_size)
    idx_full = np.sum(responses, axis=0) == responses.shape[0]
    idx_zero = np.sum(responses, axis=0) == 0
    idx_norm = ~(idx_full | idx_zero)

    # Keep the same all-correct/all-incorrect rule as EXP027 DQN.
    theta_hat[idx_full] = (
        current_theta[idx_full] + (item_bank[:, 1].max() - current_theta[idx_full]) / 2
    )
    theta_hat[idx_zero] = (
        current_theta[idx_zero] - (current_theta[idx_zero] - item_bank[:, 1].min()) / 2
    )
    if np.any(idx_norm):
        theta_hat[idx_norm] = np.squeeze(
            MLE_TEST(
                item_bank[item_ids[:, idx_norm]],
                responses[:, idx_norm],
            )
        )
    return np.clip(theta_hat, THETA_MIN, THETA_MAX)


def summarize_steps(theta_true, theta_history):
    rows = []
    theta_true_sd = np.std(theta_true, ddof=1)

    for step, theta_est in enumerate(theta_history, start=1):
        bias = theta_est - theta_true
        theta_est_sd = np.std(theta_est, ddof=1)
        correlation = (
            np.nan
            if theta_true_sd == 0 or theta_est_sd == 0
            else np.corrcoef(theta_true, theta_est)[0, 1]
        )
        rows.append(
            {
                "step": step,
                "Bias": np.mean(bias),
                "RMSE": np.sqrt(np.mean(bias**2)),
                "MAE": np.mean(np.abs(bias)),
                "r": correlation,
            }
        )

    return pd.DataFrame(rows)


def run_mfi(cfg, item_bank, response_matrix, theta_true):
    # Match the DQN TEST initial-state RNG and subject-vectorized order.
    np.random.seed(cfg.seed)
    testing_size = len(theta_true)

    theta_current = np.random.rand(testing_size) - 0.5
    item_ids = np.empty((0, testing_size), dtype=np.int64)
    responses = np.empty((0, testing_size), dtype=np.int64)
    theta_history = np.empty((0, testing_size), dtype=float)

    for step in range(cfg.test_length):
        selected = choose_mfi(item_bank, theta_current, item_ids)
        step_responses = response_matrix[np.arange(testing_size), selected]

        item_ids = np.concatenate((item_ids, selected[np.newaxis, :]))
        responses = np.concatenate((responses, step_responses[np.newaxis, :]))
        theta_current = estimate_theta_mle(
            item_bank, item_ids, responses, theta_current
        )

        theta_history = np.concatenate((theta_history, theta_current[np.newaxis, :]))
        bias = theta_current - theta_true
        print(
            "step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(
                step + 1,
                np.mean(bias),
                np.sqrt(np.mean(bias**2)),
                np.mean(np.abs(bias)),
            )
        )

    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size)
    records = pd.DataFrame(
        {
            "userID": user_id_col,
            "step": step_col,
            "itemID": (item_ids + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta_true": np.repeat(theta_true, cfg.test_length),
            "theta_est": theta_history.T.reshape(-1),
            "bias": (theta_history - theta_true).T.reshape(-1),
        }
    )
    summary_by_step = summarize_steps(theta_true, theta_history)
    return records, summary_by_step

In [11]:
cfg = Config(
    test_length=40,
    dataset="LNIRT_CredentialForm1",
    testing_size=0,
    seed=20260430,
)

data_dir = ROOT / "data" / cfg.dataset
bank_path = data_dir / "real item bank.csv"
response_path = data_dir / "real responses for testing.csv"
theta_path = data_dir / "true theta for testing.csv"

item_bank = pd.read_csv(bank_path)[["a", "b", "c"]].to_numpy()
response_matrix = pd.read_csv(response_path).to_numpy(dtype=np.int64)
theta_true = pd.read_csv(theta_path).to_numpy(dtype=float).reshape(-1)

if cfg.testing_size > 0:
    response_matrix = response_matrix[: cfg.testing_size]
    theta_true = theta_true[: cfg.testing_size]

if response_matrix.shape[1] != len(item_bank):
    raise ValueError("Response columns do not match the number of bank items.")
if len(response_matrix) != len(theta_true):
    raise ValueError("Response and true-theta row counts do not match.")
if cfg.test_length > len(item_bank):
    raise ValueError("test_length cannot exceed the number of bank items.")
if len(theta_true) < 2:
    raise ValueError("At least two examinees are required for correlation.")
if not np.isin(response_matrix, [0, 1]).all():
    raise ValueError("The response matrix must contain only 0/1 values.")

print(f"item bank : {item_bank.shape} ({bank_path})")
print(f"responses : {response_matrix.shape} ({response_path})")
print(f"theta_true: {theta_true.shape} ({theta_path})")
print(f"Config    : {cfg}")

item bank : (170, 3) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/LNIRT_CredentialForm1/real item bank.csv)
responses : (491, 170) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/LNIRT_CredentialForm1/real responses for testing.csv)
theta_true: (491,) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/LNIRT_CredentialForm1/true theta for testing.csv)
Config    : Config(test_length=40, dataset='LNIRT_CredentialForm1', testing_size=0, seed=20260430)


In [12]:
records, summary_by_step = run_mfi(cfg, item_bank, response_matrix, theta_true)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
stem = f"real_{cfg.dataset}_MFI_MLE"
records_path = RESULTS_DIR / f"records_{stem}.csv"
summary_path = RESULTS_DIR / f"summary_{stem}.csv"
records.to_csv(records_path, index=False)
summary_by_step.to_csv(summary_path, index=False)

display(summary_by_step.tail(1))
print(f"Saved records to: {records_path}")
print(f"Saved summary to: {summary_path}")

step 1, bias 0.135, rmse 1.894, mae 1.658
step 2, bias 0.091, rmse 1.482, mae 1.177
step 3, bias 0.145, rmse 1.259, mae 0.992
step 4, bias 0.154, rmse 1.101, mae 0.848
step 5, bias 0.175, rmse 1.018, mae 0.785
step 6, bias 0.199, rmse 1.020, mae 0.760
step 7, bias 0.193, rmse 0.949, mae 0.708
step 8, bias 0.171, rmse 0.890, mae 0.670
step 9, bias 0.165, rmse 0.851, mae 0.640
step 10, bias 0.143, rmse 0.791, mae 0.591
step 11, bias 0.141, rmse 0.760, mae 0.565
step 12, bias 0.124, rmse 0.728, mae 0.542
step 13, bias 0.122, rmse 0.703, mae 0.526
step 14, bias 0.107, rmse 0.686, mae 0.509
step 15, bias 0.102, rmse 0.648, mae 0.488
step 16, bias 0.101, rmse 0.634, mae 0.478
step 17, bias 0.105, rmse 0.626, mae 0.467
step 18, bias 0.109, rmse 0.617, mae 0.458
step 19, bias 0.109, rmse 0.606, mae 0.452
step 20, bias 0.101, rmse 0.581, mae 0.438
step 21, bias 0.102, rmse 0.560, mae 0.427
step 22, bias 0.095, rmse 0.543, mae 0.418
step 23, bias 0.086, rmse 0.533, mae 0.407
step 24, bias 0.080,

,step,Bias,RMSE,MAE,r
39,40,0.045418,0.399208,0.289246,0.94066


Saved records to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results/records_real_LNIRT_CredentialForm1_MFI_MLE.csv
Saved summary to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results/summary_real_LNIRT_CredentialForm1_MFI_MLE.csv
